# Training run: R(2+1)D-18 for EF regression (Kaggle)

This notebook trains the model on EchoNet-Dynamic using 2x NVIDIA T4 GPUs on Kaggle. It is kept as a record of the training run. The reusable, cleaned code lives in `src/` (`dataset.py`, `model.py`, `evaluate.py`).

- Paths in the first cell are Kaggle-specific.
- The output below is the original run log translated into English. The numbers are unchanged; the notebook was **not re-executed** after translation.
- Best validation MAE: **4.306** at epoch 18 (the checkpoint used for all results in the README).


In [1]:
# ==========================================================
# 1. Locate the dataset automatically on Kaggle
# ==========================================================
import os
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

# Working directory and output folders on Kaggle
PROJECT_DIR = Path("/kaggle/working/EchoNet_Project")
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = PROJECT_DIR / "figures"

for directory in [PROJECT_DIR, CHECKPOINT_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Search the input folder for FileList.csv
input_dir = Path("/kaggle/input")
filelist_paths = list(input_dir.rglob("FileList.csv"))

if filelist_paths:
    FILELIST_CSV = filelist_paths[0]
    DATASET_ROOT = FILELIST_CSV.parent
    VIDEOS_DIR = DATASET_ROOT / "Videos"
    
    print(f"Dataset found at: {DATASET_ROOT}")
    manifest = pd.read_csv(FILELIST_CSV)
    print(f"Videos available: {len(manifest)}.")
    display(manifest.head())
else:
    print("Error: dataset not found.")
    print("Add EchoNet-Dynamic through the 'Add Data' button in the sidebar.")

# ==========================================================
# 2. Batch size
# ==========================================================
BATCH_SIZE = 16 
NUM_FRAMES = 32
IMG_SIZE = 112

print(f"\nBatch size set to: {BATCH_SIZE}")

# ==========================================================
# 3. Prepare the model for two GPUs (T4 x2)
# ==========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()

print(f"\nGPUs available for training: {gpu_count}")

class EFRegressor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = R2Plus1D_18_Weights.DEFAULT if pretrained else None
        self.backbone = r2plus1d_18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.backbone(x)

model = EFRegressor(pretrained=True)

if gpu_count > 1:
    print(f"Multi-GPU enabled: training is distributed over {gpu_count} GPUs.")
    model = nn.DataParallel(model)

model = model.to(device)
print("Model ready for training.")


Dataset found at: /kaggle/input/datasets/mahnurrahman/echonet-dynamic/EchoNet-Dynamic
Videos available: 10030.
             FileName         EF         ESV         EDV  FrameHeight  \
0  0X100009310A3BD7FC  78.498406   14.881368   69.210534          112
1  0X1002E8FBACD08477  59.101988   40.383876   98.742884          112
2  0X1005D03EED19C65B  62.363798   14.267784   37.909734          112
3  0X10075961BC11C88E  54.545097   33.143084   72.914210          112
4  0X10094BA0A028EAC3  24.887742  127.581945  169.855024          112

   FrameWidth  FPS  NumberOfFrames  Split
0         112   50             174    VAL
1         112   50             215  TRAIN
2         112   50             104  TRAIN
3         112   55             122  TRAIN
4         112   52             207    VAL

Batch size set to: 16

GPUs available for training: 2
Multi-GPU enabled: training is distributed over 2 GPUs.
Model ready for training.


In [2]:
# ==========================================================
# 4. Full dataset and DataLoaders
# ==========================================================
import cv2
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# 1. Split the data (full dataset)
print("Splitting full dataset...")
train_df = manifest[manifest["Split"].str.upper() == "TRAIN"].reset_index(drop=True)
val_df = manifest[manifest["Split"].str.upper() == "VAL"].reset_index(drop=True)
test_df = manifest[manifest["Split"].str.upper() == "TEST"].reset_index(drop=True)

print(f"Splits -> Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")

# 2. Dataset class
MEAN = [0.43216, 0.394666, 0.37645]
STD = [0.22803, 0.22145, 0.216989]

class EchoNetDataset(Dataset):
    def __init__(self, dataframe, video_dir, num_frames=32, image_size=112, is_train=False):
        self.df = dataframe
        self.video_dir = Path(video_dir)
        self.num_frames = num_frames
        self.image_size = image_size
        self.is_train = is_train

        base_transforms = [
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ]

        if self.is_train:
            aug_transforms = [
                transforms.ToPILImage(),
                transforms.RandomRotation(15), 
                transforms.ColorJitter(brightness=0.2, contrast=0.2), 
            ]
            self.transform = transforms.Compose(aug_transforms + base_transforms)
        else:
            self.transform = transforms.Compose([transforms.ToPILImage()] + base_transforms)

    def __len__(self):
        return len(self.df)

    def sample_indices(self, total_frames):
        return np.linspace(0, max(total_frames - 1, 0), self.num_frames).astype(int)

    def read_video(self, path):
        cap = cv2.VideoCapture(str(path))
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.image_size, self.image_size))
            frames.append(frame)
        cap.release()
        return np.array(frames, dtype=np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = self.video_dir / f"{row['FileName']}.avi"
        frames = self.read_video(video_path)
        indices = self.sample_indices(len(frames))
        processed_frames = [self.transform(frames[i]) for i in indices]
        frames_tensor = torch.stack(processed_frames).permute(1, 0, 2, 3)
        ef = torch.tensor(row["EF"], dtype=torch.float32)
        return frames_tensor, ef

# 3. Create DataLoaders
train_dataset = EchoNetDataset(train_df, VIDEOS_DIR, NUM_FRAMES, IMG_SIZE, is_train=True)
val_dataset = EchoNetDataset(val_df, VIDEOS_DIR, NUM_FRAMES, IMG_SIZE, is_train=False)
test_dataset = EchoNetDataset(test_df, VIDEOS_DIR, NUM_FRAMES, IMG_SIZE, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders created successfully for the full dataset.")

Splitting full dataset...
Splits -> Train: 7465 | Validation: 1288 | Test: 1277
DataLoaders created successfully for the full dataset.


In [3]:
# ==========================================================
# 5. Training loop with smart resume
# ==========================================================
import time
import torch
from sklearn.metrics import mean_absolute_error, r2_score

# Settings
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
USE_AMP = torch.cuda.is_available()

# Optimizer, loss and scheduler
criterion = nn.SmoothL1Loss(beta=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

start_epoch = 0
best_mae = float("inf")
early_counter = 0

# --- Resume from the last checkpoint ---
checkpoint_path = CHECKPOINT_DIR / "last_checkpoint.pth"

if checkpoint_path.exists():
    print(f"\nFound a previous checkpoint at: {checkpoint_path}")
    print("Restoring weights to resume training...")
    
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # With DataParallel the state-dict keys must match
    if isinstance(model, nn.DataParallel) and not list(checkpoint["model_state_dict"].keys())[0].startswith('module.'):
        model.module.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint["model_state_dict"])
        
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    
    start_epoch = checkpoint["epoch"]
    best_mae = checkpoint.get("best_mae", float("inf"))
    
    print(f"Resuming from epoch {start_epoch + 1} (best MAE so far: {best_mae:.4f})\n")
else:
    print("\nStarting training from scratch...\n")

print("="*60)

# --- Training loop ---
for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start_time = time.time()

    # Training phase
    model.train()
    running_loss = 0.0
    for videos, targets in train_loader:
        videos = videos.to(device)
        targets = targets.float().to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            outputs = model(videos)
            loss = criterion(outputs.view(-1), targets.view(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # Validation phase
    model.eval()
    val_running_loss = 0.0
    predictions, targets_list = [], []

    with torch.no_grad():
        for videos, targets in val_loader:
            videos = videos.to(device)
            targets = targets.float().to(device)

            with torch.amp.autocast('cuda', enabled=USE_AMP):
                outputs = model(videos)
                loss = criterion(outputs.view(-1), targets.view(-1))
            val_running_loss += loss.item()

            predictions.extend(outputs.view(-1).cpu().numpy())
            targets_list.extend(targets.view(-1).cpu().numpy())

    scheduler.step()

    # Compute metrics
    val_loss = val_running_loss / len(val_loader)
    mae = mean_absolute_error(targets_list, predictions)
    r2 = r2_score(targets_list, predictions)

    # Save the current training state (for resuming later)
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_mae": best_mae,
    }, checkpoint_path)

    # Save the best model
    if mae < best_mae:
        best_mae = mae
        early_counter = 0
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_mae": best_mae,
        }, CHECKPOINT_DIR / "best_model.pth")
        print(f"[Epoch {epoch+1}] Best model updated! MAE: {best_mae:.4f}")
    else:
        early_counter += 1

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | MAE: {mae:.3f} | R²: {r2:.3f} | Time: {time.time() - epoch_start_time:.1f}s")

    if early_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs (no improvement in validation MAE).")
        break

print("=" * 60 + "\nTraining complete.")

Starting training from scratch...

[Epoch 1] Best model updated! MAE: 5.8911
Epoch [1/20] | Train Loss: 14.8962 | Val Loss: 5.3925 | MAE: 5.891 | R²: 0.607 | Time: 439.0s
Epoch [2/20] | Train Loss: 6.3078 | Val Loss: 5.6689 | MAE: 6.168 | R²: 0.519 | Time: 394.5s
Epoch [3/20] | Train Loss: 5.7919 | Val Loss: 5.8690 | MAE: 6.369 | R²: 0.536 | Time: 396.6s
Epoch [4/20] | Train Loss: 5.4903 | Val Loss: 6.7893 | MAE: 7.294 | R²: 0.442 | Time: 398.4s
[Epoch 5] Best model updated! MAE: 5.0673
Epoch [5/20] | Train Loss: 5.3475 | Val Loss: 4.5772 | MAE: 5.067 | R²: 0.695 | Time: 396.0s
Epoch [6/20] | Train Loss: 5.0944 | Val Loss: 5.1301 | MAE: 5.617 | R²: 0.645 | Time: 395.3s
[Epoch 7] Best model updated! MAE: 4.9820
Epoch [7/20] | Train Loss: 4.8954 | Val Loss: 4.4878 | MAE: 4.982 | R²: 0.692 | Time: 395.6s
[Epoch 8] Best model updated! MAE: 4.5443
Epoch [8/20] | Train Loss: 4.9050 | Val Loss: 4.0605 | MAE: 4.544 | R²: 0.746 | Time: 396.3s
Epoch [9/20] | Train Loss: 4.7339 | Val Loss: 4.4233